# Persiapan Data: Transaksi UMKM F&B (Jan–Sep 2025)
by Sujata Mulyana

**Tujuan:** menyiapkan data transaksi agar siap dipakai di dashboard Power BI.

## Sumber data

- **Dataset:** Anonymous Transactional Dataset
- **Sumber:** Mendeley Data, V2. DOI: [10.17632/kcgf45y24m.2](https://doi.org/10.17632/kcgf45y24m.2)
- **Kontributor:** Mychael Maoeretz Engel, Ford Lumban Gaol, Aditya Kurniawan, Widodo Budiharto
- **Lisensi:** [Creative Commons Attribution 4.0 International (CC BY 4.0)](https://creativecommons.org/licenses/by/4.0/)
- **Perubahan:** data asli dipecah (kolom `Items`), digabung dengan data produk, dan ditambah kolom turunan untuk analisis. Isi data tidak diubah.

## Isi notebook

1. Setup dan muat data
2. Profil data
3. Memecah kolom `Items`
4. Validasi hasil
5. Menggabungkan dengan data produk
6. Kolom tambahan untuk dashboard
7. Menyimpan hasil
8. Temuan awal
9. Kesimpulan

## 1. Setup dan muat data

In [11]:
import pandas as pd
transactions = pd.read_csv('data/raw/Transactions.csv')
products = pd.read_csv('data/raw/Products.csv')

## 2. Profil data

In [12]:
print('Transactions:', transactions.shape)
print('Products:', products.shape)

Transactions: (53820, 11)
Products: (96, 6)


In [13]:
print(transactions.dtypes)

Outlet             object
Date               object
Time               object
NetSales          float64
Tax               float64
TotalAmount         int64
ReceiptNumber      object
Items              object
TotalItem           int64
PaymentMethod      object
UseLoyaltyCard      int64
dtype: object


Ditemukan bahwa kolom date disini masih dalam bentuk string, sehingga agar kolom tersebut dapat digunakan untuk analisis, saya ubah menjadi datetime64

In [14]:
transactions['Date'] = pd.to_datetime(transactions['Date'])

In [15]:
print(transactions['Date'].dtype)

datetime64[ns]


In [16]:
print('Tanggal awal :', transactions['Date'].min().date())
print('Tanggal akhir:', transactions['Date'].max().date())
print('Hari unik    :', transactions['Date'].nunique())

Tanggal awal : 2025-01-02
Tanggal akhir: 2025-09-30
Hari unik    : 272


Hasil: 2025-01-02 sampai 2025-09-30, 272 hari unik.

In [17]:
print(transactions['Outlet'].value_counts())
print()
print(transactions['PaymentMethod'].value_counts())

Outlet
SHOP001    26250
SHOP003    16753
SHOP002    10817
Name: count, dtype: int64

PaymentMethod
BCA       42805
Cash      10913
GoFood      102
Name: count, dtype: int64


In [18]:
print('Nilai kosong Transactions:', transactions.isna().sum().sum())
print('Nilai kosong Products    :', products.isna().sum().sum())
print('ReceiptNumber ganda      :', transactions['ReceiptNumber'].duplicated().sum())
print('Baris duplikat penuh     :', transactions.duplicated().sum())

Nilai kosong Transactions: 0
Nilai kosong Products    : 0
ReceiptNumber ganda      : 0
Baris duplikat penuh     : 0


## 3. Memecah kolom `Items`

**Masalah:** satu struk bisa berisi lebih dari satu produk, dan semuanya ditulis dalam satu sel `Items`, dipisah koma. Contoh: `Friendly Coffee (Hot), Shakencano`. Untuk analisis produk, kita butuh **satu baris per produk**.

**Pemeriksaan sebelum memecah:** pemisah koma + spasi aman dipakai. Tidak ada nama produk maupun varian di `Products.csv` yang mengandung koma (0 dari 96 produk). Bagian validasi di bawah juga membuktikannya: semua hasil pecahan cocok dengan `Products.csv`.

**Metode:** `Series.str.split` untuk memecah, lalu `DataFrame.explode` supaya setiap produk menjadi satu baris. Kolom `ReceiptNumber` ikut terbawa ke setiap baris hasil pecahan.

In [20]:
transactions["Items"].head()

0                   Basic Latte (Ice Arabica)
1                          BLACK (Houseblend)
2           Friendly Coffee (Hot), Shakencano
3    BLACK (Arabica), Americano (Hot Arabica)
4                   WHITE (Arabica), Susu Oat
Name: Items, dtype: object

Terbukti jika struk lebih dari 1 produk semuanya ditulis dalam 1 sel, sehingga kita akan menggunakan str.split dan .explode dari pandas untuk memecah kolom items tersebut

In [22]:
line_items = transactions[['ReceiptNumber', 'Items']].copy()

line_items['Item'] = line_items['Items'].str.split(', ')

# explode: satu elemen daftar menjadi satu baris
line_items = line_items.explode('Item').drop(columns='Items')

# rapikan spasi dan reset index
line_items['Item'] = line_items['Item'].str.strip()
line_items = line_items.reset_index(drop=True)

print(line_items.shape)
line_items.head()

(61973, 2)


,ReceiptNumber,Item
0,SHOP001JAN2500001,Basic Latte (Ice Arabica)
1,SHOP001JAN2500002,BLACK (Houseblend)
2,SHOP002JAN2500001,Friendly Coffee (Hot)
3,SHOP002JAN2500001,Shakencano
4,SHOP002JAN2500002,BLACK (Arabica)


## 4. Validasi hasil

In [23]:
# Validasi 1: jumlah produk per struk sama dengan TotalItem
counts = line_items.groupby('ReceiptNumber').size()
expected = transactions.set_index('ReceiptNumber')['TotalItem']
assert (counts.reindex(expected.index) == expected).all()

# Validasi 2: total baris sama dengan jumlah seluruh TotalItem
assert len(line_items) == transactions['TotalItem'].sum()

# Validasi 3: semua nama produk ada di Products.csv
# (nama di transaksi = ProductName + (Variant), atau hanya ProductName kalau Variant '-')
products['Item'] = products['ProductName'].where(
    products['Variant'] == '-',
    products['ProductName'] + ' (' + products['Variant'] + ')'
)
unmatched = line_items.loc[~line_items['Item'].isin(products['Item']), 'Item']
assert unmatched.empty

print('Semua validasi lolos')
print('Produk unik di transaksi:', line_items['Item'].nunique())
print('Produk di Products.csv  :', products['Item'].nunique())

Semua validasi lolos
Produk unik di transaksi: 94
Produk di Products.csv  : 96


assert tidak mengeluarkan apa-apa apabila benar, dan memunculkan error jika salah.

Berdasarkan hasil validasi yang saya lakukan ditemukan bahwa:
1. Jumlah produk hasil pecahan per struk **sama dengan** kolom `TotalItem`.
2. Total baris `line_items` sama dengan jumlah seluruh `TotalItem`.
3. Setiap nama produk di `line_items` ada di `Products.csv` (nama produk digabung dengan varian, kecuali varian `-`).

## 5. Menggabungkan dengan data produk

Di `Items`, nama produk ditulis sebagai `ProductName (Variant)`, atau hanya `ProductName` jika varian-nya `-`.

Kolom kunci `Item` dibuat di tabel `products` pada bagian validasi, lalu digabung untuk mendapat `ProductId`, `Category`, dan `Price` tiap baris.

In [24]:
# Gabung untuk mendapat ProductId, Category, dan Price
line_items = line_items.merge(
    products[['Item', 'ProductId', 'Category', 'Price']],
    on='Item', how='left', validate='many_to_one'
)
assert line_items['Category'].notna().all()   # tidak ada produk yang gagal cocok

# Tandai struk penukaran loyalitas dan hitung pendapatan sebenarnya
is_redemption = line_items['ReceiptNumber'].map(
    transactions.set_index('ReceiptNumber')['UseLoyaltyCard']) == 1
line_items['IsRedemption'] = is_redemption
line_items['Revenue'] = line_items['Price'].where(~is_redemption, 0)

# Rekonsiliasi: pendapatan per struk harus sama dengan TotalAmount
rev = line_items.groupby('ReceiptNumber')['Revenue'].sum()
assert (rev.reindex(transactions['ReceiptNumber']).values
        == transactions['TotalAmount'].values).all()

# Produk di Products.csv yang tidak pernah muncul di transaksi
products.loc[~products['Item'].isin(line_items['Item']),
             ['ProductId', 'Item', 'Category', 'Price']]

,ProductId,Item,Category,Price
0,1,Add on Aren,Add on,0
1,2,Add on Simple Syrup,Add on,0


Kode ini berfungsi untuk menggabungkan data transaksi dengan informasi produk, menghitung revenue sebenarnya (termasuk pengecualian transaksi loyalty)

## 6. Kolom tambahan untuk dashboard

Kolom turunan ditambahkan di tabel transaksi agar dashboard lebih mudah dibuat:
- tipe tanggal yang benar, bulan, nama hari
- jam (dari kolom `Time`)
- label pelanggan loyalitas vs non-loyalitas (dari `UseLoyaltyCard`)

In [25]:
transactions['MonthNum']  = transactions['Date'].dt.month
transactions['Month']     = transactions['Date'].dt.strftime('%Y-%m')
transactions['DayOfWeek'] = transactions['Date'].dt.dayofweek   # Senin = 0, untuk mengurutkan
transactions['DayName']   = transactions['Date'].dt.day_name()
transactions['Hour']      = pd.to_datetime(transactions['Time'], format='%H:%M:%S').dt.hour
transactions['CustomerType'] = transactions['UseLoyaltyCard'].map(
    {1: 'Tukar loyalitas (gratis)', 0: 'Bayar'})

Digunakan untuk menambahkan kolom waktu dan tipe pelanggan dari data transaksi agar memudahkan analisis tren penjualan, pola waktu pembelian, serta perbandingan pelanggan loyalty dan pelanggan reguler

## 7. Menyimpan hasil

Simpan tiga file bersih untuk diimpor ke Power BI: `transactions_clean.csv`, `line_items.csv` (sudah berisi kategori dan harga), dan `products_clean.csv`.

In [26]:
transactions.to_csv('transactions_clean.csv', index=False)
line_items.to_csv('line_items.csv', index=False)
products.to_csv('products_clean.csv', index=False)

## 8. Temuan awal

Sebagai jembatan ke dashboard, dihitung ringkasan sederhana berikut:
- total penjualan dan rata-rata nilai per struk
- penjualan per bulan
- jumlah transaksi dan penjualan per outlet
- proporsi metode pembayaran
- 10 produk paling sering terjual.

In [27]:
paid = transactions[transactions['UseLoyaltyCard'] == 0]

print('Total penjualan        : Rp {:,.0f}'.format(transactions['TotalAmount'].sum()))
print('Struk berbayar         :', len(paid))
print('Struk penukaran gratis :', (transactions['UseLoyaltyCard'] == 1).sum())
print('Rata-rata per struk berbayar: Rp {:,.0f}'.format(paid['TotalAmount'].mean()))

Total penjualan        : Rp 1,315,276,000
Struk berbayar         : 52087
Struk penukaran gratis : 1733
Rata-rata per struk berbayar: Rp 25,252


In [28]:
bulanan = transactions.groupby('Month').agg(
    Penjualan=('TotalAmount', 'sum'),
    StrukBayar=('UseLoyaltyCard', lambda s: (s == 0).sum()),
    Penukaran=('UseLoyaltyCard', 'sum'))
bulanan['RataStruk'] = (bulanan['Penjualan'] / bulanan['StrukBayar']).round(0)
bulanan

,Penjualan,StrukBayar,Penukaran,RataStruk
Month,,,,
2025-01,131233000,5194,178,25266.0
2025-02,149180000,5930,194,25157.0
2025-03,152385000,6072,183,25096.0
2025-04,110195000,4295,144,25657.0
2025-05,132639000,5275,151,25145.0
2025-06,182769000,7272,258,25133.0
2025-07,176286000,6950,226,25365.0
2025-08,145624000,5765,207,25260.0
2025-09,134965000,5334,192,25303.0


In [29]:
outlet = transactions.groupby('Outlet').agg(
    Penjualan=('TotalAmount', 'sum'),
    StrukBayar=('UseLoyaltyCard', lambda s: (s == 0).sum()),
    Penukaran=('UseLoyaltyCard', 'sum'))
outlet['RataStruk'] = (outlet['Penjualan'] / outlet['StrukBayar']).round(0)
outlet['Porsi%'] = (outlet['Penjualan'] / outlet['Penjualan'].sum() * 100).round(1)
outlet

,Penjualan,StrukBayar,Penukaran,RataStruk,Porsi%
Outlet,,,,,
SHOP001,644108000,25485,765,25274.0,49.0
SHOP002,261664000,10411,406,25133.0,19.9
SHOP003,409504000,16191,562,25292.0,31.1


In [30]:
print(paid['PaymentMethod'].value_counts())
print((paid.groupby('PaymentMethod')['TotalAmount'].sum() / paid['TotalAmount'].sum() * 100).round(1))

PaymentMethod
BCA       42805
Cash       9180
GoFood      102
Name: count, dtype: int64
PaymentMethod
BCA       82.2
Cash      17.6
GoFood     0.2
Name: TotalAmount, dtype: float64


In [31]:
terjual = line_items[~line_items['IsRedemption']]

top10 = (terjual.groupby('Item')
         .agg(Terjual=('Item', 'size'), Pendapatan=('Revenue', 'sum'))
         .sort_values('Terjual', ascending=False).head(10))
display(top10)

kategori = line_items.groupby('Category')['Revenue'].sum().sort_values(ascending=False)
(kategori / kategori.sum() * 100).round(1)

,Terjual,Pendapatan
Item,,
Americano (Hot Arabica),706,14120000
BLACK (Houseblend),704,14080000
Lemons,702,14040000
Shakencano,699,14679000
Upgrade Beans (Arabica),698,2792000
Jasmine (Hot),697,14637000
Jasmine (Ice),695,14595000
Friendly Coffee (Ice),688,15824000
Americano (Ice Houesblend),686,13720000


Category
Coffee        36.0
Menu Gedhe    26.1
Other         16.4
Tea           13.2
Snacks         7.0
Add on         1.3
Name: Revenue, dtype: float64

In [32]:
print(paid.groupby('Hour')['ReceiptNumber'].count())
print(paid.groupby(['DayOfWeek', 'DayName'])['ReceiptNumber'].count())

Hour
7     2243
8     2251
9     3286
10    3035
11    3098
12    3144
13    3137
14    2981
15    3055
16    3156
17    3739
18    3739
19    3768
20    3860
21    3745
22    3850
Name: ReceiptNumber, dtype: int64
DayOfWeek  DayName  
0          Monday       7399
1          Tuesday      7451
2          Wednesday    7276
3          Thursday     7438
4          Friday       7582
5          Saturday     7346
6          Sunday       7595
Name: ReceiptNumber, dtype: int64


| Hal | Temuan |
|---|---|
| Total | Rp 1.315.276.000, 52.087 struk berbayar, 1.733 penukaran gratis, rata-rata Rp 25.252 per struk berbayar |
| Bulan | Terendah **April** (Rp 110,2 jt, 4.295 struk), tertinggi **Juni** (Rp 182,8 jt, 7.272 struk) |
| Rata-rata per struk | Stabil di Rp 25.096 sampai 25.657 tiap bulan |
| Outlet | SHOP001 49,0%, SHOP003 31,1%, SHOP002 19,9% dari penjualan; rata-rata per struk hampir sama (Rp 25.133 sampai 25.292) |
| Pembayaran | BCA 82,2% pendapatan, Cash 17,6%, GoFood 0,2% (102 struk) |
| Kategori | Coffee 36,0%, Menu Gedhe 26,1%, Other 16,4%, Tea 13,2%, Snacks 7,0%, Add on 1,3% |
| Jam | Paling ramai jam 17 sampai 22 (sekitar 3.700 sampai 3.860 struk per jam), siang 9 sampai 16 sekitar 3.000 sampai 3.300, jam 7 sampai 8 sekitar 2.250 |

## 9. Kesimpulan

### Apa yang dilakukan
Notebook ini menyiapkan data transaksi UMKM F&B (53.820 struk dari 3 outlet, 2 Januari sampai 30 September 2025). Kolom `Items`, yang berisi beberapa produk dalam satu sel, dipecah menjadi 61.973 baris produk, lalu digabung dengan data produk untuk mendapat kategori dan harga. Hasilnya divalidasi: jumlah produk per struk sesuai `TotalItem`, semua nama produk cocok dengan data produk, dan pendapatan per struk sama dengan `TotalAmount`. Tiga file bersih disimpan untuk Power BI: `transactions_clean.csv`, `line_items.csv`, dan `products_clean.csv`.

### Temuan utama
- Total penjualan **Rp 1.315.276.000** dari 52.087 struk berbayar, dengan rata-rata Rp 25.252 per struk.
- Penjualan bulanan berfluktuasi: terendah di **April** (Rp 110,2 juta) dan tertinggi di **Juni** (Rp 182,8 juta). Rata-rata per struk hampir tidak berubah, jadi naik-turunnya penjualan berasal dari **jumlah struk**, bukan dari nilai belanja per struk.
- **SHOP001** menyumbang 49,0% penjualan, **SHOP003** 31,1%, dan **SHOP002** 19,9%.
- Pembayaran didominasi **BCA** (82,2% pendapatan), diikuti tunai (17,6%). GoFood hanya 102 struk.
- Kategori terbesar adalah **Coffee** (36,0%) dan **Menu Gedhe** (26,1%).
- Jam tersibuk adalah **17.00 sampai 22.59**, sedangkan jam 7 sampai 8 paling sepi.
- Sebanyak **1.733 struk** yang memakai kartu loyalitas bernilai Rp 0, hanya berisi 1 produk, dan dibayar tunai. Pola ini menunjukkan penukaran hadiah gratis (dugaan, belum dikonfirmasi), sehingga struk ini dihitung terpisah dari penjualan berbayar.

### Keterbatasan
- Harga produk berasal dari `Products.csv`, bukan dari transaksi. Untuk semua struk berbayar, jumlah harga produk sama persis dengan `TotalAmount`, sehingga pendapatan per produk dapat diandalkan.
- Nama outlet dianonimkan (SHOP001 sampai SHOP003), jadi tidak bisa dikaitkan dengan lokasi atau karakter outlet.
- Penjualan antar produk dan antar hari dalam seminggu **sangat merata**, lebih merata daripada yang lazim pada kafe sungguhan. Perbedaan antar kategori dan antar jam lebih bermakna daripada peringkat produk. Penyebab pola ini tidak dapat dipastikan dari data.
- Penyebab rendahnya penjualan April tidak dapat ditentukan dari data ini dan memerlukan konteks dari pemilik usaha.
- Data hanya mencakup 9 bulan, sehingga pola musiman tahunan belum dapat dinilai.
- Tidak ada data biaya, sehingga laba tidak dapat dianalisis.

### Langkah berikutnya
Impor ketiga file bersih ke Power BI dan bangun dashboard dua halaman: ringkasan bisnis (KPI, tren bulanan, outlet, pembayaran) serta kategori dan jam sibuk.